In [1]:
import numpy as np
import pandas as pd
import os
from datetime import datetime, date
from statsmodels.tsa.vector_ar.var_model import VAR
from statsmodels.tsa.vector_ar.vecm import VECM, coint_johansen
import scipy.linalg as la
from scipy.linalg import eig
from statsmodels.tsa.vector_ar.var_model import VAR
import statsmodels.api as sm
from statsmodels.tsa.stattools import adfuller
print(f"modules have been imported.")

modules have been imported.


In [2]:
# Step 1: Lets grab the historical time series from Mohaddes first. 
# Question should we grab it by economies first? if yes then lets take the 'Country Data'
print(os.getcwd())
target_filename = "Country Data (1979Q2-2023Q3).xls"
target_file_path = os.getcwd() + "\\" + "GVAR Database (1979Q2-2023Q3)\\" + target_filename
print(f"target_file_path is: {target_file_path}")
# dataframe the country data
countries_df = pd.read_excel(target_file_path, sheet_name = None)
countries_df.keys()


C:\Users\Ben Jin
target_file_path is: C:\Users\Ben Jin\GVAR Database (1979Q2-2023Q3)\Country Data (1979Q2-2023Q3).xls


dict_keys(['ARGENTINA', 'AUSTRALIA', 'BRAZIL', 'CANADA', 'CHINA', 'CHILE', 'EURO', 'INDIA', 'INDONESIA', 'JAPAN', 'KOREA', 'MALAYSIA', 'MEXICO', 'NORWAY', 'NEW ZEALAND', 'PERU', 'PHILIPPINES', 'SOUTH AFRICA', 'SAUDI ARABIA', 'SINGAPORE', 'SWEDEN', 'SWITZERLAND', 'THAILAND', 'TURKEY', 'UNITED KINGDOM', 'USA'])

In [3]:
us_df = countries_df['USA']
sg_df = countries_df['SINGAPORE']
cn_df = countries_df['CHINA']
eu_df = countries_df['EURO']
# convert the date into datetime format 
# ie: 1979Q2 is 30-Jun-1979.
us_df['date_2'] = pd.PeriodIndex(us_df['date'], freq = 'Q').to_timestamp(how = "end").date
sg_df['date_2'] = pd.PeriodIndex(sg_df['date'], freq = 'Q').to_timestamp(how = "end").date
cn_df['date_2'] = pd.PeriodIndex(cn_df['date'], freq = 'Q').to_timestamp(how = "end").date
eu_df['date_2'] = pd.PeriodIndex(eu_df['date'], freq = 'Q').to_timestamp(how = "end").date


us_target_cols = [col for col in us_df.columns if 'date' not in col]
us_target_cols = ['date_2'] + us_target_cols 

sg_target_cols = [col for col in sg_df.columns if 'date' not in col]
sg_target_cols = ['date_2'] + sg_target_cols 

cn_target_cols = [col for col in cn_df.columns if 'date' not in col]
cn_target_cols = ['date_2'] + cn_target_cols 

eu_target_cols = [col for col in eu_df.columns if 'date' not in col]
eu_target_cols = ['date_2'] + eu_target_cols 

print(us_target_cols)

us_df = us_df[us_target_cols]
us_df = us_df.rename(columns={'date_2': 'date'})
us_df

sg_df = sg_df[sg_target_cols]
sg_df = sg_df.rename(columns={'date_2': 'date'})
sg_df

cn_df = cn_df[cn_target_cols]
cn_df = cn_df.rename(columns={'date_2': 'date'})
cn_df

eu_df = eu_df[eu_target_cols]
eu_df = eu_df.rename(columns={'date_2': 'date'})
eu_df

['date_2', 'y', 'Dp', 'eq', 'r', 'lr', 'ys', 'Dps', 'eps', 'poil', 'pmat', 'pmetal']


,date,y,Dp,eq,ep,r,lr,ys,Dps,eqs,rs,lrs,poil,pmat,pmetal
0,1979-06-30,4.151148,0.022753,0.701674,-4.156059,0.020858,0.023323,3.763988,0.032185,0.977932,0.022302,0.023070,3.433544,4.148960,4.200168
1,1979-09-30,4.156398,0.025299,0.723492,-4.217551,0.022919,0.024231,3.771680,0.043473,0.956643,0.023587,0.023496,3.576322,4.272871,4.198063
2,1979-12-31,4.166828,0.026988,0.662959,-4.257434,0.025633,0.025393,3.785490,0.034167,0.922622,0.025699,0.026133,3.696729,4.220098,4.305733
3,1980-03-31,4.176844,0.032215,0.672845,-4.271255,0.027541,0.027575,3.793602,0.045573,0.923275,0.027895,0.028130,3.661658,4.311326,4.431295
4,1980-06-30,4.171065,0.024655,0.649555,-4.298715,0.029884,0.028336,3.787795,0.040089,0.916227,0.026489,0.026966,3.643353,4.207531,4.284341
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
173,2022-09-30,4.815708,0.020158,1.990970,-5.561242,0.000891,0.005307,5.337170,0.018444,2.696272,0.009416,0.006349,4.613407,4.814593,5.240803
174,2022-12-31,4.810953,0.021251,2.095719,-5.569188,0.003811,0.007124,5.338953,0.014712,2.736946,0.011651,0.007851,4.485170,4.719217,5.244402
175,2023-03-31,4.813094,0.009064,2.195740,-5.528113,0.006374,0.007455,5.347404,0.014027,2.764883,0.012978,0.007611,4.397714,4.696997,5.416306
176,2023-06-30,4.821536,0.007371,2.201635,-5.520525,0.007670,0.007590,5.356329,0.011225,2.781942,0.014519,0.007856,4.361766,4.674107,5.315188


In [4]:
# 1. We first and foremostly must do is to do the ADF (augmented dickey fuller) test to check on its unit root testing. 
## 2. We need to do the AIC and BIC test to find out the lags that are most optimal for the VARX
## 3. This is to test the weak exogeneity for each variable components in the us_df table. 

def run_gvar_stationary_check(country, dataframe, significance_level = 0.05):
    adf_results = []
    # 2. Automated ADF testing loop function
    for column in dataframe.columns:
        if column not in 'date':
            
            # --- TEST 1: Check Raw Levels ---
            # autolag = 'AIC' lets the module find the best lag length to remove serial autocorrelation.
            # regression = 'c' includes a constant intercept in the test equation.
            # let significance_level = 0.05
            level_res = adfuller(dataframe[column], autolag = 'AIC', regression = 'c')
            level_pvalue = level_res[1]
            level_stationary = level_pvalue < significance_level
            
#            print(f"--- Results for the US {column} ---")
#            print(f"level_res is: {level_res}")
#            print(f"level_pvalue is: {level_pvalue}")
#            print(f"level_stationary is: {level_stationary}")
            
            # Determine Integration Order Classification
            if not level_stationary:
                classification = "I(1) - Perfect for VECM"
            elif level_stationary:
                classification = "I(0) - Already Stationary"
            else:
                classification = "I(2) or Higher/ Explosive"
            
            adf_results.append({
                'country' : country,
                'variable' : column,
                'level_pvalue' : round(level_pvalue, 4),
                'level_stationary' : level_stationary,
                'classification' : classification,
            })
        
    return pd.DataFrame(adf_results)

us_adfuller_df = run_gvar_stationary_check("US", us_df, significance_level = 0.05)
sg_adfuller_df = run_gvar_stationary_check("SG", sg_df, significance_level = 0.05)
cn_adfuller_df = run_gvar_stationary_check("CN", cn_df, significance_level = 0.05)
eu_adfuller_df = run_gvar_stationary_check("EU", eu_df, significance_level = 0.05)

all_adfuller_df = us_adfuller_df.copy()
all_adfuller_df = pd.concat([all_adfuller_df, sg_adfuller_df, cn_adfuller_df, eu_adfuller_df], axis = 0)

display(all_adfuller_df)

,country,variable,level_pvalue,level_stationary,classification
0,US,y,0.6722,False,I(1) - Perfect for VECM
1,US,Dp,0.0000,True,I(0) - Already Stationary
2,US,eq,0.7832,False,I(1) - Perfect for VECM
3,US,r,0.0901,False,I(1) - Perfect for VECM
4,US,lr,0.5487,False,I(1) - Perfect for VECM
5,US,ys,0.3974,False,I(1) - Perfect for VECM
6,US,Dps,0.6558,False,I(1) - Perfect for VECM
7,US,eps,0.8429,False,I(1) - Perfect for VECM
8,US,poil,0.6556,False,I(1) - Perfect for VECM
9,US,pmat,0.1268,False,I(1) - Perfect for VECM


# The code below is supposed to be a module to find the best lag order based on AIC, BIC and HQIC. 
# However, seeing that we need to see the results of the VECMX. I will get back to this when i can have a better picture on how the lags p, q = 2,2 works. 

In [5]:
# this code block will be used for the AIC, BIC and HQIC

endog_vars = ['y', 'Dp', 'eq', 'r', 'lr', 'poil', 'pmat', 'pmetal']
exog_vars = ['ys', 'Dps']

us_endog_df = us_df[endog_vars]
us_exog_df = us_df[exog_vars]

display(us_endog_df)
display(us_exog_df)

# define variables
exog_df = us_exog_df
endog_df = us_endog_df
#display(exog_df)
#print(f"endog_df is: \n {display(endog_df)}")

# Grid parameters
max_p = 3 # for domestic/ endog vars
max_q = 2 # for foreign/ exog vars
max_lag = max(max_p, max_q)

results = []

# 2. High level optimization loop using statsmodels internals. 
for p in range(1, max_p + 1):
    for q in range(0, max_q + 1):
        
        print(f"p,q are: ({p},{q})")
        # Construct foreign variables lag columns
        exog_list = []
        for lag in range(0, q + 1):
            print(f"lag (q) now is: {lag}")
            target_exog_df = exog_df.shift(lag)
            if lag >= 1:
                target_exog_df.columns = [f"{col}_lag{q}" for col in target_exog_df.columns]

            exog_list.append(target_exog_df) # shift the lag based on q variable.      
        
        X_exog = pd.concat(exog_list, axis = 1).dropna() # to concatenate by columns. 
#        print(f"X_exog is:\n {display(X_exog)}")
#        print(f"X_exog.index is:\n {display(X_exog.index)}")

        # Align endog target dataframe to match exogenous shift row losses.
        Y_target = endog_df.loc[X_exog.index]
#        print(f"Y_target is:\n {display(Y_target)}")

        # CRUCIAL GVAR TRIMMING: Ensure ALL lag combinations run on the exact same time window. 
        # This keeps the comparison mathematically fair across the loops. 
        X_exog_clean = X_exog.iloc[(max_lag - p):] # max_lag = 3, p = 1, ans = 2 
        Y_target_clean = Y_target.iloc[(max_lag - p):] # max_lag = 3, p = 1, ans = 2

#        print(f"X_exog_clean is:\n {display(X_exog_clean)}")
        print(f"X_exog_clean shape is:\n {(X_exog_clean.shape)}")
#        print(f"Y_target_clean is:\n {display(Y_target_clean)}")
        print(f"Y_target_clean shape is:\n {(Y_target_clean.shape)}")

        # 3. Call the built-in statsmodels VAR function with exog parameters
        # maxlags = p forces the engine to fit exactly 'p' for domestic lags
        model = VAR(endog = Y_target_clean, exog = X_exog_clean)
        results_fitted = model.fit(maxlags=p)

        # 4. Pull the native optimized AIC/BIC/HQIC calculations from the results object.
        results.append({
            'p' : p,
            'q' : q,
            'AIC' : results_fitted.aic,
            'BIC' : results_fitted.bic,
            'HQIC' : results_fitted.hqic,
        })
        
#        if q == 1: 
#            print(f"lets look at the results first.")
#            break

# 5. Display the final score sheet.
df_scores = pd.DataFrame(results)
print("--- Statsmodels Evaluated Scores ---")
print(f"df_scores is: \n {display(df_scores)}")
print(df_scores.to_string(index=False))

# Extract the winning combinations
print(f"\nWinning Architecture by AIC: P={df_scores.loc[df_scores['AIC'].idxmin(), 'p']}, Q = {df_scores.loc[df_scores['AIC'].idxmin(), 'q']}")
print(f"\nWinning Architecture by BIC: P={df_scores.loc[df_scores['BIC'].idxmin(), 'p']}, Q = {df_scores.loc[df_scores['BIC'].idxmin(), 'q']}")
print(f"\nWinning Architecture by HQIC: P={df_scores.loc[df_scores['HQIC'].idxmin(), 'p']}, Q = {df_scores.loc[df_scores['HQIC'].idxmin(), 'q']}")

,y,Dp,eq,r,lr,poil,pmat,pmetal
0,3.960503,0.033730,0.684670,0.022407,0.021804,3.433544,4.148960,4.200168
1,3.967677,0.032477,0.699070,0.023084,0.021781,3.576322,4.272871,4.198063
2,3.970622,0.028823,0.640953,0.027982,0.024841,3.696729,4.220098,4.305733
3,3.973806,0.038217,0.615193,0.031335,0.028302,3.661658,4.311326,4.431295
4,3.953413,0.035451,0.609975,0.022955,0.024909,3.643353,4.207531,4.284341
...,...,...,...,...,...,...,...,...
173,5.008861,0.014444,2.906448,0.006571,0.007648,4.613407,4.814593,5.240803
174,5.015219,0.009122,2.961698,0.009901,0.009396,4.485170,4.719217,5.244402
175,5.018379,0.010911,3.020958,0.011307,0.008954,4.397714,4.696997,5.416306
176,5.029266,0.004715,3.095903,0.012372,0.008826,4.361766,4.674107,5.315188


,ys,Dps
0,3.723439,0.027880
1,3.736803,0.029817
2,3.755427,0.029535
3,3.766682,0.040626
4,3.768565,0.033697
...,...,...
173,5.331930,0.014947
174,5.332679,0.011538
175,5.343477,0.008314
176,5.352381,0.007976


p,q are: (1,0)
lag (q) now is: 0
X_exog_clean shape is:
 (176, 2)
Y_target_clean shape is:
 (176, 8)
p,q are: (1,1)
lag (q) now is: 0
lag (q) now is: 1
X_exog_clean shape is:
 (175, 4)
Y_target_clean shape is:
 (175, 8)
p,q are: (1,2)
lag (q) now is: 0
lag (q) now is: 1
lag (q) now is: 2
X_exog_clean shape is:
 (174, 6)
Y_target_clean shape is:
 (174, 8)
p,q are: (2,0)
lag (q) now is: 0
X_exog_clean shape is:
 (177, 2)
Y_target_clean shape is:
 (177, 8)
p,q are: (2,1)
lag (q) now is: 0
lag (q) now is: 1
X_exog_clean shape is:
 (176, 4)
Y_target_clean shape is:
 (176, 8)
p,q are: (2,2)
lag (q) now is: 0
lag (q) now is: 1
lag (q) now is: 2
X_exog_clean shape is:
 (175, 6)
Y_target_clean shape is:
 (175, 8)
p,q are: (3,0)
lag (q) now is: 0
X_exog_clean shape is:
 (178, 2)
Y_target_clean shape is:
 (178, 8)
p,q are: (3,1)
lag (q) now is: 0
lag (q) now is: 1
X_exog_clean shape is:
 (177, 4)
Y_target_clean shape is:
 (177, 8)
p,q are: (3,2)
lag (q) now is: 0
lag (q) now is: 1
lag (q) now is:

,p,q,AIC,BIC,HQIC
0,1,0,-67.836095,-66.244659,-67.190563
1,1,1,-69.152206,-67.264035,-68.386248
2,1,2,-69.424765,-67.237511,-68.537409
3,2,0,-68.243021,-65.494179,-67.128013
4,2,1,-69.267610,-66.217488,-68.030293
5,2,2,-69.620710,-66.266920,-68.260098
6,3,0,-68.192103,-64.285853,-66.607616
7,3,1,-69.262275,-65.050201,-67.553599
8,3,2,-69.663047,-65.142721,-67.829178


df_scores is: 
 None
 p  q        AIC        BIC       HQIC
 1  0 -67.836095 -66.244659 -67.190563
 1  1 -69.152206 -67.264035 -68.386248
 1  2 -69.424765 -67.237511 -68.537409
 2  0 -68.243021 -65.494179 -67.128013
 2  1 -69.267610 -66.217488 -68.030293
 2  2 -69.620710 -66.266920 -68.260098
 3  0 -68.192103 -64.285853 -66.607616
 3  1 -69.262275 -65.050201 -67.553599
 3  2 -69.663047 -65.142721 -67.829178

Winning Architecture by AIC: P=3, Q = 2

Winning Architecture by BIC: P=1, Q = 1

Winning Architecture by HQIC: P=1, Q = 2


$ \Large\text{We are working on the VECMX* (1,1) based from the VARX(2,2)} $

$ \Large\text{Here is the formula below so that we can understand which formulas we are going to keep as levels (contemporary and lagged) and change in variables (contemporary and lagged.)} $

$ \Large\text{VARX* is:  } x_{i,t} = a_{i,0} + \Phi_{i,1} x_{i, t-1} + \Phi_{i,2} x_{i,t-2} + \Lambda_{i,0} x^*_{i,t} + \Lambda_{i,1} x^*_{i, t-1} + \Lambda_{i,2} x^*_{i,t-2} + \epsilon_{i,t} $

$ \Large\text{VECMX* is:  } \Delta x_{i,t} = a_{i,0} + \Gamma_{i,1} \Delta x_{i, t-1} + \Psi_{i,0} \Delta x^*_{i,t} + \Psi_{i,1} \Delta x_{i,t-1} + \Pi_{i} z_{i,t-1} + \epsilon_{i,t} $

$ \Large\text{where: } $

$ \Large\text{1. } \Delta x_{i,t} = x_{i,t} - x_{i,t-1} $

$ \Large\text{2. } \Delta x_{i,t-1} = x_{i,t-1} - x_{i,t-2} $

$ \Large\text{3. } \Delta z_{i,t-1} = \begin{bmatrix} x_{i,t-1} \\ x^*_{i,t-1} \end{bmatrix} $

In [6]:
# ok so for the US model below, lets follow the simple example of VARX(2,2) and VECMX(1,1)
# p, q = 2, 2

# Defining the variables for domestic vs. foreign.
endog_vars = ['y', 'Dp', 'eq', 'r', 'lr', 'poil', 'pmat', 'pmetal']
exog_vars = ['ys', 'Dps']

us_endog_df = us_df[endog_vars]
us_exog_df = us_df[exog_vars]

display(us_endog_df)
#display(us_exog_df)

# lets assume that our lags are as follows p_{domestic} = 2, q_{foreign} = 2
# so we are now finding for VAR(p, q) = VAR(2, 2)
# -- USER DEFINED PARAMETERS -- # 
# Once when we are done with the testing we can remove the user defined parameters and use the AIC test to indicate the lags
p, q = 2, 2

max_lag = max(p,q)

# to define the exog data here based on lags (defined by AIC/BIC/HQIC) and shifts (based on also lags?).
us_exog_contemp_df = us_exog_df.iloc[max_lag:] # align to match the loss of p domestic lags.
us_exog_lag1_df = us_exog_df.shift(1).iloc[max_lag:].add_suffix('_lag1') # lag levels of 1
us_exog_lag2_df = us_exog_df.shift(2).iloc[max_lag:].add_suffix('_lag2')  # lag levels of 2
print(f"us_exog_contemp_df is: \n {display(us_exog_contemp_df)}")
print(f"us_exog_lag1_df is: \n {display(us_exog_lag1_df)}")
print(f"us_exog_lag2_df is: \n {display(us_exog_lag2_df)}")

#print(f"us_exog_df shift(1) is: {display(us_exog_df.shift(1))}")
#print(f"us_exog_df shift(1).iloc[p:] is: {display(us_exog_df.shift(1).iloc[p:])}")
#print(f"us_exog_df shift(2) is: {display(us_exog_df.shift(2))}")
#print(f"us_exog_df shift(2).iloc[p:] is: {display(us_exog_df.shift(2).iloc[p:])}")

# Concatenate the exog data here with the shifts and lags. 
us_exog_final_df = pd.concat([us_exog_contemp_df, us_exog_lag1_df, us_exog_lag2_df], axis = 1)
print(f"us_exog_final_df is: {display(us_exog_final_df)}")

# Add in the constant from the exog data. 
us_exog_final_df = sm.add_constant(us_exog_final_df)
print(f"us_exog_final_df after including constant is: {display(us_exog_final_df)}")

# define us endogeneous data
us_endog_final_df = us_endog_df.iloc[p:]
print(f"us_endog_final_df is: \n {display(us_endog_final_df)}")

#--- VECMX Data Prep ---#
# This is where we switch it from VARX(2,2) = VECMX(1,1)
# lets compute the \delta x_{i,t} = x_{i,t} - x_{i,t-1} --> dependent variable 
us_endog_vecm_comtemp_df = us_df[endog_vars]
us_endog_vecm_lag1_df = us_df[endog_vars].shift(1).iloc[max_lag:].add_suffix('_lag1')
us_endog_vecm_lag2_df = us_df[endog_vars].shift(2).iloc[max_lag:].add_suffix('_lag2')
us_endog_vecm_diff1_df = us_df[endog_vars].diff().add_suffix('_diff1')
us_endog_vecm_diff2_df = us_df[endog_vars].shift(1).diff().add_suffix('_diff2')

print(f"us_endog_vecm_comtemp_df is: \n {display(us_endog_vecm_comtemp_df)}")
print(f"us_endog_vecm_lag1_df is: \n {display(us_endog_vecm_lag1_df)}")
print(f"us_endog_vecm_lag2_df is: \n {display(us_endog_vecm_lag2_df)}")
print(f"us_endog_vecm_diff1_df is: \n {display(us_endog_vecm_diff1_df)}")
print(f"us_endog_vecm_diff2_df is: \n {display(us_endog_vecm_diff2_df)}")

us_endog_vecm_final_df = pd.concat([us_endog_vecm_comtemp_df, us_endog_vecm_lag1_df, us_endog_vecm_lag2_df, us_endog_vecm_diff1_df, us_endog_vecm_diff2_df], axis = 1)
print(f"us_endog_vecm_df is: \n {display(us_endog_vecm_final_df)}")

us_exog_final_df = pd.concat([us_exog_contemp_df, us_exog_lag1_df, us_exog_lag2_df], axis = 1)

us_final_vecm_df = pd.concat([us_endog_vecm_final_df, us_exog_final_df], axis = 1).dropna()
print(f"us_final_vecm_df is: \n {display(us_final_vecm_df)}")

# sort the columns
us_final_vecm_df = us_final_vecm_df.sort_index(axis = 1)
print(f"us_final_vecm_df after sorting is: \n {display(us_final_vecm_df)}")

# ---- TO NOTE: WE DO NOT NEED TO RUN VARX FIRST, WE CAN GO AHEAD AND DO VECMX FIRST. ----- #
# Fit the level VARX model via statsmodels 
# we used maxlags = max(p,q) for the domestic variables and passing foreign vectors as exog.
varx_model = VAR(endog = us_endog_final_df, exog = us_exog_final_df)
varx_results = varx_model.fit(maxlags = p)

print(f"--- US VARX({p},{q}) Model Estimated Successfully ---")
print(f"Endogenous Matrix Dimensions (Variables): {us_endog_final_df.shape[1]}")
print(f"Exogenous Matrix Dimensions (Regressors): {us_exog_final_df.shape[1]}")
print(f"\nCoefficient Parameters shape for the US system:", varx_results.params.shape)
print(varx_results.summary())

,y,Dp,eq,r,lr,poil,pmat,pmetal
0,3.960503,0.033730,0.684670,0.022407,0.021804,3.433544,4.148960,4.200168
1,3.967677,0.032477,0.699070,0.023084,0.021781,3.576322,4.272871,4.198063
2,3.970622,0.028823,0.640953,0.027982,0.024841,3.696729,4.220098,4.305733
3,3.973806,0.038217,0.615193,0.031335,0.028302,3.661658,4.311326,4.431295
4,3.953413,0.035451,0.609975,0.022955,0.024909,3.643353,4.207531,4.284341
...,...,...,...,...,...,...,...,...
173,5.008861,0.014444,2.906448,0.006571,0.007648,4.613407,4.814593,5.240803
174,5.015219,0.009122,2.961698,0.009901,0.009396,4.485170,4.719217,5.244402
175,5.018379,0.010911,3.020958,0.011307,0.008954,4.397714,4.696997,5.416306
176,5.029266,0.004715,3.095903,0.012372,0.008826,4.361766,4.674107,5.315188


,ys,Dps
2,3.755427,0.029535
3,3.766682,0.040626
4,3.768565,0.033697
5,3.776422,0.032422
6,3.787429,0.032043
...,...,...
173,5.331930,0.014947
174,5.332679,0.011538
175,5.343477,0.008314
176,5.352381,0.007976


us_exog_contemp_df is: 
 None


,ys_lag1,Dps_lag1
2,3.736803,0.029817
3,3.755427,0.029535
4,3.766682,0.040626
5,3.768565,0.033697
6,3.776422,0.032422
...,...,...
173,5.320754,0.021158
174,5.331930,0.014947
175,5.332679,0.011538
176,5.343477,0.008314


us_exog_lag1_df is: 
 None


...

us_endog_vecm_df is: 
 None


,y,Dp,eq,r,lr,poil,pmat,pmetal,y_lag1,Dp_lag1,...,lr_diff2,poil_diff2,pmat_diff2,pmetal_diff2,ys,Dps,ys_lag1,Dps_lag1,ys_lag2,Dps_lag2
2,3.970622,0.028823,0.640953,0.027982,0.024841,3.696729,4.220098,4.305733,3.967677,0.032477,...,-0.000023,0.142778,0.123911,-0.002106,3.755427,0.029535,3.736803,0.029817,3.723439,0.027880
3,3.973806,0.038217,0.615193,0.031335,0.028302,3.661658,4.311326,4.431295,3.970622,0.028823,...,0.003059,0.120407,-0.052773,0.107670,3.766682,0.040626,3.755427,0.029535,3.736803,0.029817
4,3.953413,0.035451,0.609975,0.022955,0.024909,3.643353,4.207531,4.284341,3.973806,0.038217,...,0.003462,-0.035071,0.091228,0.125562,3.768565,0.033697,3.766682,0.040626,3.755427,0.029535
5,3.951749,0.018735,0.694921,0.021896,0.025985,3.550055,4.289884,4.277811,3.953413,0.035451,...,-0.003394,-0.018305,-0.103795,-0.146954,3.776422,0.032422,3.768565,0.033697,3.766682,0.040626
6,3.970122,0.025836,0.724808,0.031908,0.029275,3.679337,4.284165,4.195716,3.951749,0.018735,...,0.001076,-0.093299,0.082353,-0.006530,3.787429,0.032043,3.776422,0.032422,3.768565,0.033697
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
173,5.008861,0.014444,2.906448,0.006571,0.007648,4.613407,4.814593,5.240803,5.000886,0.021992,...,0.002416,0.123919,0.058115,-0.029523,5.331930,0.014947,5.320754,0.021158,5.317501,0.016353
174,5.015219,0.009122,2.961698,0.009901,0.009396,4.485170,4.719217,5.244402,5.008861,0.014444,...,0.000429,-0.119741,-0.135785,-0.220728,5.332679,0.011538,5.331930,0.014947,5.320754,0.021158
175,5.018379,0.010911,3.020958,0.011307,0.008954,4.397714,4.696997,5.416306,5.015219,0.009122,...,0.001748,-0.128237,-0.095376,0.003599,5.343477,0.008314,5.332679,0.011538,5.331930,0.014947
176,5.029266,0.004715,3.095903,0.012372,0.008826,4.361766,4.674107,5.315188,5.018379,0.010911,...,-0.000442,-0.087456,-0.022220,0.171903,5.352381,0.007976,5.343477,0.008314,5.332679,0.011538


us_final_vecm_df is: 
 None


,Dp,Dp_diff1,Dp_diff2,Dp_lag1,Dp_lag2,Dps,Dps_lag1,Dps_lag2,eq,eq_diff1,...,r_lag1,r_lag2,y,y_diff1,y_diff2,y_lag1,y_lag2,ys,ys_lag1,ys_lag2
2,0.028823,-0.003654,-0.001252,0.032477,0.033730,0.029535,0.029817,0.027880,0.640953,-0.058117,...,0.023084,0.022407,3.970622,0.002945,0.007175,3.967677,3.960503,3.755427,3.736803,3.723439
3,0.038217,0.009394,-0.003654,0.028823,0.032477,0.040626,0.029535,0.029817,0.615193,-0.025760,...,0.027982,0.023084,3.973806,0.003184,0.002945,3.970622,3.967677,3.766682,3.755427,3.736803
4,0.035451,-0.002766,0.009394,0.038217,0.028823,0.033697,0.040626,0.029535,0.609975,-0.005218,...,0.031335,0.027982,3.953413,-0.020394,0.003184,3.973806,3.970622,3.768565,3.766682,3.755427
5,0.018735,-0.016716,-0.002766,0.035451,0.038217,0.032422,0.033697,0.040626,0.694921,0.084946,...,0.022955,0.031335,3.951749,-0.001663,-0.020394,3.953413,3.973806,3.776422,3.768565,3.766682
6,0.025836,0.007101,-0.016716,0.018735,0.035451,0.032043,0.032422,0.033697,0.724808,0.029887,...,0.021896,0.022955,3.970122,0.018373,-0.001663,3.951749,3.953413,3.787429,3.776422,3.768565
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
173,0.014444,-0.007547,-0.001222,0.021992,0.023214,0.014947,0.021158,0.016353,2.906448,-0.066698,...,0.002677,0.000765,5.008861,0.007974,-0.001445,5.000886,5.002331,5.331930,5.320754,5.317501
174,0.009122,-0.005322,-0.007547,0.014444,0.021992,0.011538,0.014947,0.021158,2.961698,0.055251,...,0.006571,0.002677,5.015219,0.006358,0.007974,5.008861,5.000886,5.332679,5.331930,5.320754
175,0.010911,0.001789,-0.005322,0.009122,0.014444,0.008314,0.011538,0.014947,3.020958,0.059260,...,0.009901,0.006571,5.018379,0.003160,0.006358,5.015219,5.008861,5.343477,5.332679,5.331930
176,0.004715,-0.006196,0.001789,0.010911,0.009122,0.007976,0.008314,0.011538,3.095903,0.074945,...,0.011307,0.009901,5.029266,0.010887,0.003160,5.018379,5.015219,5.352381,5.343477,5.332679


us_final_vecm_df after sorting is: 
 None
--- US VARX(2,2) Model Estimated Successfully ---
Endogenous Matrix Dimensions (Variables): 8
Exogenous Matrix Dimensions (Regressors): 6

Coefficient Parameters shape for the US system: (23, 8)
  Summary of Regression Results   
Model:                         VAR
Method:                        OLS
Date:           Tue, 21, Jul, 2026
Time:                     13:28:37
--------------------------------------------------------------------
No. of Equations:         8.00000    BIC:                   -65.9430
Nobs:                     174.000    HQIC:                  -67.9285
Log likelihood:           4236.51    FPE:                8.23965e-31
AIC:                     -69.2836    Det(Omega_mle):     3.05192e-31
--------------------------------------------------------------------
Results for equation y
               coefficient       std. error           t-stat            prob
--------------------------------------------------------------------------

In [27]:
# Use Johansens test to find the cointegration (Beta) and speed of adjustment (alpha)
# lets remove the global variables from the us_df data so that we can only find the cointegration 
# relationship between the domestic vars and its foreign vars.

print(us_df.columns)

endog_vars = ['y', 'Dp', 'eq', 'r', 'lr']
exog_vars = ['ys', 'Dps', 'eps']

us_raw_level_df = pd.concat([us_df[endog_vars], us_df[exog_vars]], axis = 1)
display(us_raw_level_df)
print(f"columns for us_raw_level_df is: {us_raw_level_df.columns}")

# Run the automated Johansens test
# det_order = 0, includes an unrestricted constant intercept in the VECM space. 
# k_ar_diff = 1, enforces 1 lag of short run differences (corresponds to p = 2 for levels).
johansen_res = coint_johansen(us_raw_level_df, det_order = 0, k_ar_diff = 0)

# Rearrange the data to compare the johansens trace stats vs the crit values properly.
ci_level_df = pd.DataFrame(johansen_res.cvt, index = [i for i in range(len(johansen_res.cvt))], columns = ['ci_90%', 'ci_95%', 'ci_99%'])
trace_stats_df = pd.DataFrame(johansen_res.lr1, columns = ['trace_stats'])
max_eigenvalues_df = pd.DataFrame(johansen_res.lr2, columns = ['max_eigenvalue'])
johansen_df = pd.concat([max_eigenvalues_df, trace_stats_df, ci_level_df], axis = 1)
johansen_df['id_90%'] = np.where(johansen_df['trace_stats'] >= johansen_df['ci_90%'],1,0)
johansen_df['id_95%'] = np.where(johansen_df['trace_stats'] >= johansen_df['ci_95%'],1,0)
johansen_df['id_99%'] = np.where(johansen_df['trace_stats'] >= johansen_df['ci_99%'],1,0)
johansen_df['rank_90%'] = johansen_df['id_90%'].cumsum()
johansen_df['rank_95%'] = johansen_df['id_95%'].cumsum()
johansen_df['rank_99%'] = johansen_df['id_99%'].cumsum()


# Extracts the output from the results of the Johansen test. 
print("--- Automation Johansen Statistics ---")
print("Trace Statistics: \n", johansen_res.lr1)
print("\nMax Eigenvalue Statistics:\n", johansen_res.lr2)
print("Critical values (90%, 95%, 99% thresholds):\n", johansen_res.cvt)
print(f"final table for the johansen statistics to see the ranking is as below:")
print(display(johansen_df))

Index(['date', 'y', 'Dp', 'eq', 'r', 'lr', 'ys', 'Dps', 'eps', 'poil', 'pmat',
       'pmetal'],
      dtype='str')


,y,Dp,eq,r,lr,ys,Dps,eps
0,3.960503,0.033730,0.684670,0.022407,0.021804,3.723439,0.027880,-2.381442
1,3.967677,0.032477,0.699070,0.023084,0.021781,3.736803,0.029817,-2.420323
2,3.970622,0.028823,0.640953,0.027982,0.024841,3.755427,0.029535,-2.440535
3,3.973806,0.038217,0.615193,0.031335,0.028302,3.766682,0.040626,-2.465478
4,3.953413,0.035451,0.609975,0.022955,0.024909,3.768565,0.033697,-2.498637
...,...,...,...,...,...,...,...,...
173,5.008861,0.014444,2.906448,0.006571,0.007648,5.331930,0.014947,-3.370713
174,5.015219,0.009122,2.961698,0.009901,0.009396,5.332679,0.011538,-3.365033
175,5.018379,0.010911,3.020958,0.011307,0.008954,5.343477,0.008314,-3.395617
176,5.029266,0.004715,3.095903,0.012372,0.008826,5.352381,0.007976,-3.403638


columns for us_raw_level_df is: Index(['y', 'Dp', 'eq', 'r', 'lr', 'ys', 'Dps', 'eps'], dtype='str')
--- Automation Johansen Statistics ---
Trace Statistics: 
 [261.23338409 126.80606136  87.46034748  52.93295576  34.71010667
  16.68711516   4.90165438   0.93369163]

Max Eigenvalue Statistics:
 [134.42732273  39.34571388  34.52739172  18.22284909  18.02299151
  11.78546078   3.96796275   0.93369163]
Critical values (90%, 95%, 99% thresholds):
 [[153.6341 159.529  171.0905]
 [120.3673 125.6185 135.9825]
 [ 91.109   95.7542 104.9637]
 [ 65.8202  69.8189  77.8202]
 [ 44.4929  47.8545  54.6815]
 [ 27.0669  29.7961  35.4628]
 [ 13.4294  15.4943  19.9349]
 [  2.7055   3.8415   6.6349]]
final table for the johansen statistics to see the ranking is as below:


C:\Users\Ben Jin\AppData\Local\Programs\Python\Python313\Lib\site-packages\statsmodels\tsa\vector_ar\vecm.py:731: ComplexWarning: Casting complex values to real discards the imaginary part
  lr1[i] = -t * np.sum(tmp, 0)
C:\Users\Ben Jin\AppData\Local\Programs\Python\Python313\Lib\site-packages\statsmodels\tsa\vector_ar\vecm.py:732: ComplexWarning: Casting complex values to real discards the imaginary part
  lr2[i] = -t * np.log(1 - a[i])


,max_eigenvalue,trace_stats,ci_90%,ci_95%,ci_99%,id_90%,id_95%,id_99%,rank_90%,rank_95%,rank_99%
0,134.427323,261.233384,153.6341,159.5290,171.0905,1,1,1,1,1,1
1,39.345714,126.806061,120.3673,125.6185,135.9825,1,1,0,2,2,1
2,34.527392,87.460347,91.1090,95.7542,104.9637,0,0,0,2,2,1
3,18.222849,52.932956,65.8202,69.8189,77.8202,0,0,0,2,2,1
4,18.022992,34.710107,44.4929,47.8545,54.6815,0,0,0,2,2,1
5,11.785461,16.687115,27.0669,29.7961,35.4628,0,0,0,2,2,1
6,3.967963,4.901654,13.4294,15.4943,19.9349,0,0,0,2,2,1
7,0.933692,0.933692,2.7055,3.8415,6.6349,0,0,0,2,2,1


None


,ci_90%,ci_95%,ci_99%
0,153.6341,159.5290,171.0905
1,120.3673,125.6185,135.9825
2,91.1090,95.7542,104.9637
3,65.8202,69.8189,77.8202
4,44.4929,47.8545,54.6815
5,27.0669,29.7961,35.4628
6,13.4294,15.4943,19.9349
7,2.7055,3.8415,6.6349


,trace_stats
0,261.233384
1,126.806061
2,87.460347
3,52.932956
4,34.710107
5,16.687115
6,4.901654
7,0.933692


,max_eigenvalue,trace_stats,ci_90%,ci_95%,ci_99%
0,134.427323,261.233384,153.6341,159.5290,171.0905
1,39.345714,126.806061,120.3673,125.6185,135.9825
2,34.527392,87.460347,91.1090,95.7542,104.9637
3,18.222849,52.932956,65.8202,69.8189,77.8202
4,18.022992,34.710107,44.4929,47.8545,54.6815
5,11.785461,16.687115,27.0669,29.7961,35.4628
6,3.967963,4.901654,13.4294,15.4943,19.9349
7,0.933692,0.933692,2.7055,3.8415,6.6349


,max_eigenvalue,trace_stats,ci_90%,ci_95%,ci_99%,id_90%,id_95%,id_99%,rank_90%,rank_95%,rank_99%
0,134.427323,261.233384,153.6341,159.5290,171.0905,1,1,1,1,1,1
1,39.345714,126.806061,120.3673,125.6185,135.9825,1,1,0,2,2,1
2,34.527392,87.460347,91.1090,95.7542,104.9637,0,0,0,2,2,1
3,18.222849,52.932956,65.8202,69.8189,77.8202,0,0,0,2,2,1
4,18.022992,34.710107,44.4929,47.8545,54.6815,0,0,0,2,2,1
5,11.785461,16.687115,27.0669,29.7961,35.4628,0,0,0,2,2,1
6,3.967963,4.901654,13.4294,15.4943,19.9349,0,0,0,2,2,1
7,0.933692,0.933692,2.7055,3.8415,6.6349,0,0,0,2,2,1
